# Kurs: [Docker und Kubernetes – Übersicht und Einsatz ](https://www.digicomp.ch/trends/docker-trainings/docker-und-kubernetes-uebersicht-und-einsatz)

Dieses Notebook dient als strukturiertes Hilfsmittel zur Analyse und zum Troubleshooting der Kursumgebung.

Es enthält vorbereitete Abfragen und Korrekturschritte, mit denen typische Probleme (z. B. Deployment-Fehler, Ressourcenengpässe oder Konfigurationsabweichungen) systematisch identifiziert und behoben werden können.


---
### Übungen nachführen

In [ ]:
%%bash
DIRECTORIES=("duk" "k8sd2o" "llmeng" "terra" "platen")

for dir in "${DIRECTORIES[@]}"
do
    if [ -d "$HOME/$dir" ]
    then
        echo "Verzeichnis '$dir' gefunden. führe Update aus..."
        git -C "$HOME/$dir" fetch --all
        git -C "$HOME/$dir" reset --hard @{u}
    fi
done

---
### Fehlende Software nachinstallieren

In [ ]:
! kubectl get ns "cert-manager" >/dev/null 2>&1 || curl -sfL https://raw.githubusercontent.com/mc-b/lerncloud/main/services/cert-manager.sh | bash -

In [ ]:
! kubectl get ns "opentelemetry" >/dev/null 2>&1 || curl -sfL https://raw.githubusercontent.com/mc-b/lerncloud/main/services/opentelemetry.sh | bash -

---
### Installation mittels Cloud-init

Für die Installation wurde Cloud-init verwendet und folgende Deklaration

In [ ]:
%%bash
sudo sudo cat /var/lib/cloud/instance/user-data.txt

Mittels des Cloud-init Logs kann die Installation auf etwaige Fehler überprüft werden, ebenfalls erhalten wir eine Übersicht über die installierten Module

In [ ]:
%%bash
sudo cat /var/log/cloud-init-output.log | grep -E 'INFO|ERROR' || true


- - - 

Ist alles korrekt durchgelaufen, kann anhand der Systemdienste (linux daemons) die Distribution erkannt werden:

In [ ]:
%%bash
systemctl list-units --type=service | grep -E 'microk8s|k3s|k0s' || echo "Keine K8s Distribution gefunden"

---

### Kein Zugriff auf Kubernetes Cluster

Bei fehlendem Zugriff auf ein **MicroK8s**-Cluster liegt die Ursache häufig in einer fehlenden oder veralteten Kubernetes-Konfiguration.

In diesem Fall kann die Konfiguration neu erzeugt und lokal gespeichert werden:

In [ ]:
%%bash
sudo microk8s config | tee ~/.kube/config

In [ ]:
%%bash
kubectl get nodes -o wide

---

### Server-IP

Die nachfolgende Zelle ermittelt automatisch die IP-Adresse oder den DNS-Namen des aktuellen Servers.

Unterstützt werden sowohl lokale Umgebungen (z. B. Hyper-V, Multipass) als auch Cloud-Plattformen wie AWS, Azure und Google Cloud. Abhängig von der erkannten Umgebung wird die passende Methode zur Ermittlung verwendet.

Wird keine bekannte Umgebung erkannt, erfolgt die Bestimmung der IP-Adresse über die aktive Netzwerkschnittstelle des Servers.

Das Ergebnis (IP-Adresse oder DNS-Name) wird in der Datei `~/data/server-ip` gespeichert und kann von anderen Notebooks weiterverwendet werden.


In [ ]:
%%bash
# Public IP anhand Cloud Provider setzen, WireGuard ueberschreibt alle
curl -fsSL https://raw.githubusercontent.com/mc-b/lerncloud/main/scripts/get-server-ip.sh | bash | tee ~/data/server-ip

---
### Headlamp

Das Headlamp ist eine moderne alternative Weboberfläche für Kubernetes. Im Gegensatz zum klassischen Kubernetes Dashboard fokussiert sich Headlamp stärker auf Übersichtlichkeit, Multi-Cluster-Unterstützung und eine benutzerfreundliche Navigation.

In [ ]:
%%bash
echo "HeadLamp: http://"$(cat ~/data/server-ip)":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h

---
### Kubernetes Dashboard (retired)

Das Kubernetes Dashboard ist eine webbasierte Verwaltungsoberfläche für Kubernetes-Cluster. Es dient primär dazu, Cluster-Ressourcen visuell zu überwachen und administrative Aufgaben ohne direkte kubectl-Kommandos auszuführen

In [ ]:
! echo "https://"$(cat ~/data/server-ip)":30443"

- - -

### UUID

Die nachfolgende Zeile generiert eine eindeutige UUID (Universally Unique Identifier).

Die UUID dient als unverwechselbare Kennung, die sicherstellt, dass jede Installation eindeutig identifiziert werden kann. 

Damit kann, z.B. für die IoT Übungen ein eindeutiges MQTT Topic bereitgestellt werden.

Die generierte UUID wird im Python-Dateiformat uuid.py im Verzeichnis ~/data/ gespeichert und wird in den Python-Skripten weiterverwendet.

In [ ]:
%%bash
echo "UUID=\"$(uuid)\"" | tee ~/data/uuid.py

- - - 

### Docker Hub - Pull-Raten-Limits

Docker Hub verwendet IP-Adressen, um die Benutzer zu authentifizieren, und Pull-Raten-Limits basieren auf einzelnen IP-Adressen. 

Für **anonyme Benutzer** ist das Ratenlimit auf 100 Abrufe pro 6 Stunden pro IP-Adresse festgelegt.

Die aktuellen Zugriff können wir wie folgt abfragen:

In [ ]:
%%bash
TOKEN=$(curl -s "https://auth.docker.io/token?service=registry.docker.io&scope=repository:ratelimitpreview/test:pull" | jq -r .token)
curl --head -H "Authorization: Bearer $TOKEN" https://registry-1.docker.io/v2/ratelimitpreview/test/manifests/latest